We will describe a sample supervised learning problem in detail: the problem of deciding whether to wait for a table at a restaurant. This problem will be used throughout the chapter to demonstrate different model classes. For this problem the output, $y$, is a Boolean variable that we will call $WillWait$; it is true for examples where we do wait for a table. The input, $x$, is a vector of ten attribute values, each of which has discrete values:

1. Alternate: whether there is a suitable alternative restaurant nearby.
2. Bar: whether the restaurant has a comfortable bar area to wait in.
3. Fri/Sat: true on Fridays and Saturdays.
4. Hungry: whether we are hungry right now.
5. Patrons: how many people are in the restaurant (values are None, Some, and Full).
6. Price: the restaurant’s price range ($, $$, $$$).
7. Raining: whether it is raining outside.
8. Reservation: whether we made a reservation.
9. Type: the kind of restaurant (French, Italian, Thai, or burger).
10. WaitEstimate: host’s wait estimate: 0–10, 10–30, 30–60, or >60 minutes.

A set of 12 examples, taken from the experience of one of us (SR), is shown in Figure 19.2. Note how skimpy these data are: there are $2^6 \times 3^2 \times 4^2 = 9,216$ possible combinations of values for the input attributes, but we are given the correct output for only 12 of them; each of the other 9,204 could be either true or false; we don’t know. This is the essence of induction: we need to make our best guess at these missing 9,204 output values, given only the evidence of the 12 examples.

In [1]:
# Como representar os dados do dataset:
# Atributos como uma lista:
atributos = ['Alt', 'Bar', 'Fri', 'Hun', 'Pat', 'Pric', 'Rain', 'Res', 'Type', 'Est']
# As amostras como uma lista de dicionários com os 10 atributos como chaves e mais a chave WillWait:
# x1 Yes No No Yes Some $$$ No Yes French 0–10 y1 = Yes
# x2 Yes No No Yes Full $ No No Thai 30–60 y2 = No
# x3 No Yes No No Some $ No No Burger 0–10 y3 = Yes
# x4 Yes No Yes Yes Full $ Yes No Thai 10–30 y4 = Yes
# x5 Yes No Yes No Full $$$ No Yes French >60 y5 = No
# x6 No Yes No Yes Some $$ Yes Yes Italian 0–10 y6 = Yes
# x7 No Yes No No None $ Yes No Burger 0–10 y7 = No
# x8 No No No Yes Some $$ Yes Yes Thai 0–10 y8 = Yes
# x9 No Yes Yes No Full $ Yes No Burger >60 y9 = No
# x10 Yes Yes Yes Yes Full $$$ No Yes Italian 10–30 y10 = No
# x11 No No No No None $ No No Thai 0–10 y11 = No
# x12 Yes Yes Yes Yes Full $ No No Burger 30–60 y12 = Yes
amostras = [ 
    { 'Alt':True,  'Bar':False, 'Fri':False, 'Hun':True,  'Pat':'Some', 'Pric':'$$$', 'Rain':False, 'Res':True,  'Type':'French',  'Est':'0–10',  'WillWait':True},
    { 'Alt':True,  'Bar':False, 'Fri':False, 'Hun':True,  'Pat':'Full', 'Pric':'$',   'Rain':False, 'Res':False, 'Type':'Thai',    'Est':'30–60', 'WillWait':False},
    { 'Alt':False, 'Bar':True,  'Fri':False, 'Hun':False, 'Pat':'Some', 'Pric':'$',   'Rain':False, 'Res':False, 'Type':'Burger',  'Est':'0–10',  'WillWait':True},  
    { 'Alt':True,  'Bar':False, 'Fri':True,  'Hun':True,  'Pat':'Full', 'Pric':'$',   'Rain':True,  'Res':False, 'Type':'Thai',    'Est':'10–30', 'WillWait':True},
    { 'Alt':True,  'Bar':False, 'Fri':True,  'Hun':False, 'Pat':'Full', 'Pric':'$$$', 'Rain':False, 'Res':True,  'Type':'French',  'Est':'>60',   'WillWait':False},
    { 'Alt':False, 'Bar':True,  'Fri':False, 'Hun':True,  'Pat':'Some', 'Pric':'$$',  'Rain':True,  'Res':True,  'Type':'Italian', 'Est':'0–10',  'WillWait':True},
    { 'Alt':False, 'Bar':True,  'Fri':False, 'Hun':False, 'Pat':'None', 'Pric':'$',   'Rain':True,  'Res':False, 'Type':'Burger',  'Est':'0–10',  'WillWait':False},
    { 'Alt':False, 'Bar':False, 'Fri':False, 'Hun':True,  'Pat':'Some', 'Pric':'$$',  'Rain':True,  'Res':True,  'Type':'Thai',    'Est':'0–10',  'WillWait':True},
    { 'Alt':False, 'Bar':True,  'Fri':True,  'Hun':False, 'Pat':'Full', 'Pric':'$',   'Rain':True,  'Res':False, 'Type':'Burger',  'Est':'>60',   'WillWait':False},
    { 'Alt':True,  'Bar':True,  'Fri':True,  'Hun':True,  'Pat':'Full', 'Pric':'$$$', 'Rain':False, 'Res':True,  'Type':'Italian', 'Est':'10–30', 'WillWait':False},
    { 'Alt':False, 'Bar':False, 'Fri':False, 'Hun':False, 'Pat':'None', 'Pric':'$',   'Rain':False, 'Res':False, 'Type':'Thai',    'Est':'0–10',  'WillWait':False},
    { 'Alt':True,  'Bar':True,  'Fri':True,  'Hun':True,  'Pat':'Full', 'Pric':'$',   'Rain':False, 'Res':False, 'Type':'Burger',  'Est':'30–60', 'WillWait':True}
]   


In [2]:
"""
function LEARN-DECISION-TREE(examples, attributes, parent examples) returns a tree
    if examples is empty then return PLURALITY-VALUE(parent examples)
    else if all examples have the same classification then return the classification
    else if attributes is empty then return PLURALITY-VALUE(examples)
    else
        A←argmaxa∈attributes IMPORTANCE(a,examples)
        tree←a new decision tree with root test A
        for each value v of A do
            exs←{e : e∈examples and e.A= v}
            subtree←LEARN-DECISION-TREE(exs, attributes−A, examples)
            add a branch to tree with label (A= v) and subtree subtree
        return tree
"""

def plurality_value(examples):
    count_true = sum(1 for e in examples if e['WillWait'])
    count_false = len(examples) - count_true
    return count_true >= count_false

def learn_decision_tree(examples, attributes, parent_examples=[]):
    if not examples:
        return plurality_value(parent_examples)
    
    if all(e['WillWait'] == examples[0]['WillWait'] for e in examples):
        return examples[0]['WillWait']
    
    if not attributes:
        return plurality_value(examples)
    
    # Placeholder for IMPORTANCE function
    def importance(attribute, examples):
        # This function should compute the importance of the attribute
        # For simplicity, we will return a random value here
        return sum(1 for e in examples if e[attribute])
    
    A = max(attributes, key=lambda a: importance(a, examples))
    tree = { 'attribute': A, 'branches': {} }
    
    values = set(e[A] for e in examples)
    for v in values:
        exs = [e for e in examples if e[A] == v]
        subtree = learn_decision_tree(exs, [attr for attr in attributes if attr != A], examples)
        tree['branches'][v] = subtree
    
    return tree

In [3]:
def print_tree(tree, depth=0):
    if isinstance(tree, bool):
        print("  " * depth + f"Leaf: {tree}")
        return
    print("  " * depth + f"[Attribute: {tree['attribute']}]")
    for value, subtree in tree['branches'].items():
        print("  " * (depth + 1) + f"(Value: {value})")
        print_tree(subtree, depth + 2)

In [4]:
dt_01 = learn_decision_tree(amostras, atributos)
print_tree(dt_01)

[Attribute: Pat]
  (Value: Full)
    [Attribute: Pric]
      (Value: $$$)
        Leaf: False
      (Value: $)
        [Attribute: Type]
          (Value: Burger)
            [Attribute: Bar]
              (Value: True)
                [Attribute: Fri]
                  (Value: True)
                    [Attribute: Est]
                      (Value: >60)
                        Leaf: False
                      (Value: 30–60)
                        Leaf: True
          (Value: Thai)
            [Attribute: Alt]
              (Value: True)
                [Attribute: Hun]
                  (Value: True)
                    [Attribute: Est]
                      (Value: 10–30)
                        Leaf: True
                      (Value: 30–60)
                        Leaf: False
  (Value: Some)
    Leaf: True
  (Value: None)
    Leaf: False


The decision tree learning algorithm chooses the attribute with the highest IMPORTANCE. We will now show how to measure importance, using the notion of information gain, which is defined in terms of entropy, which is the fundamental quantity in information theory (Shannon and Weaver, 1949).

Entropy is a measure of the uncertainty of a random variable; the more information, the less entropy. A random variable with only one possible value — a coin that always comes up heads — has no uncertainty and thus its entropy is defined as zero. A fair coin is equally
likely to come up heads or tails when flipped, and we will soon show that this counts as “1 bit” of entropy. The roll of a fair four-sided die has 2 bits of entropy, because there are 22 equally probable choices. Now consider an unfair coin that comes up heads 99% of the time.
Intuitively, this coin has less uncertainty than the fair coin — if we guess heads we’ll be wrong only 1% of the time — so we would like it to have an entropy measure that is close to zero, but positive. In general, the entropy of a random variable V with values vk having probability $P(v_k)$ is defined as

$$\text{Entropy: } H(v) = \sum_{k} P(v_k) \log_2 \frac{1}{P(v_k)} = -\sum_{k} P(v_k) \log_2 P(v_k).$$

We can check that the entropy of a fair coin flip is indeed 1 bit:

$$ H(Fair)= -(0.5\log_2{0.5}+0.5\log_2{0.5})=1 $$

And of a four-sided die is 2 bits:

$$H(Die4)= -(0.25\log_2{0.25}+0.25\log_2{0.25}+0.25\log_2{0.25}+0.25\log_2{0.25})=2$$

For the loaded coin with 99% heads, we get

$$ H(Loaded)= -(0.99\log_2{0.99}+0.01\log_2{0.01})\approx 0.08 $$

It will help to define $B(q)$ as the entropy of a Boolean random variable that is true with probability $q$:

$$ B(q)= -(q\log_2q + (1-q)\log_2{(1-q)}).$$

Thus, $H(Loaded) = B(0.99) ≈0.08$. Now let’s get back to decision tree learning. If a training set contains $p$ positive examples and $n$ negative examples, then the entropy of the output variable on the whole set is

$$ H(Output) = B\left (\frac{p}{p+n}\right )$$

The restaurant training set in Figure 19.2 has $p= n = 6$, so the corresponding entropy is $B(0.5)$ or exactly 1 bit. The result of a test on an attribute $A$ will give us some information, thus reducing the overall entropy by some amount. We can measure this reduction by looking
at the entropy remaining after the attribute test.

An attribute $A$ with d distinct values divides the training set $E$ into subsets $E_1,...,E_d$. Each subset $E_k$ has $p_k$ positive examples and $n_k$ negative examples, so if we go along that branch, we will need an additional $B(p_k/(p_k + n_k))$ bits of information to answer the question. A randomly chosen example from the training set has the $k$th value for the attribute (i.e., is in $E_k$ with probability $(p_k + n_k)/(p + n)$), so the expected entropy remaining after testing attribute $A$ is

$$Rimainder(A) = \sum_{k=1}^d{\frac{p_k+n_k}{p+n}B\left ( \frac{p_k}{p_k+n_k}\right )}$$

The information gain from the attribute test on $A$ is the expected reduction in entropy:

$$ Gain(A) = B(\frac{p}{p+n}) - Remainder(A) $$

In [5]:
# função para calcular o ganho de informação de uma variável com base no cálculo entropia da informação 

def information_gain(attribute, examples):
    from math import log2

    def entropy(examples):
        total = len(examples)
        if total == 0:
            return 0
        count_true = sum(1 for e in examples if e['WillWait'])
        count_false = total - count_true
        p_true = count_true / total
        p_false = count_false / total
        ent = 0
        if p_true > 0:
            ent -= p_true * log2(p_true)
        if p_false > 0:
            ent -= p_false * log2(p_false)
        return ent

    total_entropy = entropy(examples)
    values = set(e[attribute] for e in examples)
    weighted_entropy = 0
    total = len(examples)

    for v in values:
        subset = [e for e in examples if e[attribute] == v]
        weighted_entropy += (len(subset) / total) * entropy(subset)

    return total_entropy - weighted_entropy

In [6]:
gain_pat = information_gain('Pat', amostras)
print(f"Ganho de informação para o atributo 'Pat': {gain_pat}")

Ganho de informação para o atributo 'Pat': 0.5408520829727552


In [7]:
gain_type = information_gain('Type', amostras)
print(f"Ganho de informação para o atributo 'Type': {gain_type}")

Ganho de informação para o atributo 'Type': 0.0


In [8]:
def learn_decision_tree(examples, attributes, parent_examples=[], importance=information_gain):
    if not examples:
        return plurality_value(parent_examples)
    
    if all(e['WillWait'] == examples[0]['WillWait'] for e in examples):
        return examples[0]['WillWait']
    
    if not attributes:
        return plurality_value(examples)
    
    A = max(attributes, key=lambda a: importance(a, examples))
    tree = { 'attribute': A, 'branches': {} }
    
    values = set(e[A] for e in examples)
    for v in values:
        exs = [e for e in examples if e[A] == v]
        subtree = learn_decision_tree(exs, [attr for attr in attributes if attr != A], examples)
        tree['branches'][v] = subtree
    
    return tree

In [9]:
dt_02 = learn_decision_tree(amostras, atributos)
print_tree(dt_02)

[Attribute: Pat]
  (Value: Full)
    [Attribute: Hun]
      (Value: False)
        Leaf: False
      (Value: True)
        [Attribute: Type]
          (Value: Burger)
            Leaf: True
          (Value: Italian)
            Leaf: False
          (Value: Thai)
            [Attribute: Fri]
              (Value: False)
                Leaf: False
              (Value: True)
                Leaf: True
  (Value: Some)
    Leaf: True
  (Value: None)
    Leaf: False


In [10]:
dt_02

{'attribute': 'Pat',
 'branches': {'Full': {'attribute': 'Hun',
   'branches': {False: False,
    True: {'attribute': 'Type',
     'branches': {'Burger': True,
      'Italian': False,
      'Thai': {'attribute': 'Fri', 'branches': {False: False, True: True}}}}}},
  'Some': True,
  'None': False}}